In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, recall_score, precision_score

# Load Extracted Features
train_df = pd.read_csv('train_features.csv')
test_df = pd.read_csv('test_features.csv')

X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values
X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values

#  Data Normalization
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#  Kernel Comparison Suite
kernel_configs = {
    'linear': dict(kernel='linear', C=1.0),
    'poly': dict(kernel='poly', C=1.0, degree=2),
    'rbf': dict(kernel='rbf', C=15.0, gamma=0.05),
    'sigmoid': dict(kernel='sigmoid', C=1.0, gamma=0.005),
}

results = {}

for knl, params in kernel_configs.items():
    model = SVC(**params, probability=True, random_state=42)
    model.fit(X_train_scaled, y_train)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    threshold = 0.6637
    preds = (probs >= threshold).astype(int)

    results[knl] = {
        'p': precision_score(y_test, preds),
        'r': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds)
    }

#  Determine Best Kernel
best_kernel = max(results, key=lambda k: results[k]['f1'])

# Output
print("--- Multi-Kernel Analysis (Replication of Base Paper) ---")
for knl in kernel_configs:
    m = results[knl]
    print(f"{knl.upper():<10} | F1: {m['f1']:.4f} | P: {m['p']:.4f} | R: {m['r']:.4f}")

print(f"\nBest Performing Kernel: {best_kernel.upper()}")

#  RBF final result (paper's proposed model)
final = results['rbf']
print(f"\n--- Proposed Model (RBF) Results ---")
print(f"Precision : {final['p']:.4f}")
print(f"Recall : {final['r']:.4f}")
print(f"F1-Score : {final['f1']:.4f}")

--- Multi-Kernel Analysis (Replication of Base Paper) ---
LINEAR     | F1: 0.7928 | P: 0.8148 | R: 0.7719
POLY       | F1: 0.8018 | P: 0.8053 | R: 0.7982
RBF        | F1: 0.8158 | P: 0.8158 | R: 0.8158
SIGMOID    | F1: 0.8122 | P: 0.8087 | R: 0.8158

Best Performing Kernel: RBF

--- Proposed Model (RBF) Results ---
Precision : 0.8158
Recall : 0.8158
F1-Score : 0.8158


In [ ]:
rbf_model = SVC(kernel='rbf', C=15.0, gamma=0.05, probability=True, random_state=42)
rbf_model.fit(X_train_scaled, y_train)

# Select ONLY ONE Test Sample (Index 0)
sample_index = 0
x_single = X_test_scaled[sample_index]
y_single = y_test[sample_index]

# Get the decision score (distance from hyperplane) for this sample
decision_score = rbf_model.decision_function([x_single])

# Save the files
np.savetxt("support_vectors_rbf.txt", rbf_model.support_vectors_)
np.savetxt("dual_coeff_rbf.txt", rbf_model.dual_coef_)
np.savetxt("intercept_rbf.txt", rbf_model.intercept_)

np.savetxt("xtest_rbf.txt", x_single)
np.savetxt("ytest_rbf.txt", [y_single])
np.savetxt("yclassificationscore.txt", decision_score)

print("\n[System] RBF model parameters and test sample saved successfully.")


[System] RBF model parameters and test sample saved successfully.
